In [2]:
import pandas as pd
import numpy as np

1. Data Load

In [3]:
df = pd.read_csv('data/US_Accidents_March23.csv', low_memory=False)
print(f"Raw shape: {df.shape}")

Raw shape: (7728394, 46)


In [6]:
df.head(2)

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day


In [7]:
df.describe().round(2)

,Severity,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in)
count,7728394.00,7728394.00,7728394.00,4325632.00,4325632.00,7728394.00,7564541.00,5729375.00,7554250.00,7587715.00,7551296.00,7157161.00,5524808.00
mean,2.21,36.20,-94.70,36.26,-95.73,0.56,61.66,58.25,64.83,29.54,9.09,7.69,0.01
std,0.49,5.08,17.39,5.27,18.11,1.78,19.01,22.39,22.82,1.01,2.69,5.42,0.11
min,1.00,24.55,-124.62,24.57,-124.55,0.00,-89.00,-89.00,1.00,0.00,0.00,0.00,0.00
25%,2.00,33.40,-117.22,33.46,-117.75,0.00,49.00,43.00,48.00,29.37,10.00,4.60,0.00
50%,2.00,35.82,-87.77,36.18,-88.03,0.03,64.00,62.00,67.00,29.86,10.00,7.00,0.00
75%,2.00,40.08,-80.35,40.18,-80.25,0.46,76.00,75.00,84.00,30.03,10.00,10.40,0.00
max,4.00,49.00,-67.11,49.08,-67.11,441.75,207.00,207.00,100.00,58.63,140.00,1087.00,36.47


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7728394 entries, 0 to 7728393
Data columns (total 46 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ID                     object 
 1   Source                 object 
 2   Severity               int64  
 3   Start_Time             object 
 4   End_Time               object 
 5   Start_Lat              float64
 6   Start_Lng              float64
 7   End_Lat                float64
 8   End_Lng                float64
 9   Distance(mi)           float64
 10  Description            object 
 11  Street                 object 
 12  City                   object 
 13  County                 object 
 14  State                  object 
 15  Zipcode                object 
 16  Country                object 
 17  Timezone               object 
 18  Airport_Code           object 
 19  Weather_Timestamp      object 
 20  Temperature(F)         float64
 21  Wind_Chill(F)          float64
 22  Humidity(%)       

2. COLUMN SELECTION

In [29]:
keep_cols = [
    'ID', 'Severity',
    'Start_Time', 'End_Time',
    'Start_Lat', 'Start_Lng',
    'City', 'County', 'State',
    'Temperature(F)', 'Humidity(%)', 'Visibility(mi)',
    'Weather_Condition', 'Wind_Speed(mph)',
    'Junction', 'Traffic_Signal', 'Crossing', 'Roundabout',
    'Sunrise_Sunset', 'Civil_Twilight',
    'Distance(mi)'
]
df = df[keep_cols].copy()

 3. MISSING VALUE HANDLING

In [30]:
critical_cols = ['ID', 'Severity', 'Start_Time', 'Start_Lat', 'Start_Lng', 'State']
df.dropna(subset=critical_cols, inplace=True)

In [31]:
df['Temperature(F)'] = df['Temperature(F)'].fillna(df['Temperature(F)'].median())
df['Humidity(%)'].fillna(df['Humidity(%)'].median(), inplace=True)
df['Visibility(mi)'].fillna(df['Visibility(mi)'].median(), inplace=True)
df['Wind_Speed(mph)'].fillna(df['Wind_Speed(mph)'].median(), inplace=True)
df['Weather_Condition'].fillna('Unknown', inplace=True)
df['Sunrise_Sunset'].fillna('Unknown', inplace=True)

C:\Users\user\AppData\Local\Temp\ipykernel_20476\3724771917.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Humidity(%)'].fillna(df['Humidity(%)'].median(), inplace=True)
C:\Users\user\AppData\Local\Temp\ipykernel_20476\3724771917.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 

In [ ]:
bool_cols = ['Junction', 'Traffic_Signal', 'Crossing', 'Roundabout']
df[bool_cols] = df[bool_cols].fillna(False) 

4. DATA TYPES

In [33]:
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
df['End_Time']   = pd.to_datetime(df['End_Time'],   errors='coerce')
df.dropna(subset=['Start_Time'], inplace=True) 

5. REMOVE DUPLICATES

In [34]:
df.drop_duplicates(subset='ID', keep='first', inplace=True)

6. FEATURE ENGINEERING

In [35]:
df['Hour']       = df['Start_Time'].dt.hour
df['DayOfWeek']  = df['Start_Time'].dt.day_name()   # "Monday", "Tuesday" …
df['Month']      = df['Start_Time'].dt.month_name()  # "January" …
df['Year']       = df['Start_Time'].dt.year
df['Is_Weekend'] = df['Start_Time'].dt.dayofweek.isin([5, 6])


month_order = {m: i for i, m in enumerate(
    ['January','February','March','April','May','June',
     'July','August','September','October','November','December'], 1)}
df['Month_Num'] = df['Month'].map(month_order)


day_order = {'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,
             'Friday':5,'Saturday':6,'Sunday':7}
df['DayOfWeek_Num'] = df['DayOfWeek'].map(day_order)

In [36]:
df['Duration_Min'] = (df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60
df['Duration_Min'] = df['Duration_Min'].clip(lower=0, upper=1440)
df.loc[df['Duration_Min'] == 0, 'Duration_Min'] = np.nan  # 0-min = unreliable


In [37]:
def time_bucket(hour):
    if 6 <= hour < 10:   return 'AM Rush (6–10)'
    if 10 <= hour < 16:  return 'Midday (10–16)'
    if 16 <= hour < 20:  return 'PM Rush (16–20)'
    if 20 <= hour < 24:  return 'Evening (20–24)'
    return 'Night (0–6)'
df['Time_Bucket'] = df['Hour'].apply(time_bucket)

df['Is_Rush_Hour'] = df['Time_Bucket'].isin(['AM Rush (6–10)', 'PM Rush (16–20)'])

In [38]:
df['Severity_Group'] = df['Severity'].apply(
    lambda x: 'High (3–4)' if x >= 3 else 'Low (1–2)'
)


In [39]:
def group_weather(w):
    w = str(w).lower()
    if any(k in w for k in ['snow', 'sleet', 'ice', 'blizzard', 'wintry']):
        return 'Snow/Ice'
    if any(k in w for k in ['fog', 'mist', 'haze', 'smoke', 'smog']):
        return 'Fog/Haze'
    if any(k in w for k in ['rain', 'drizzle', 'shower', 'thunder', 'storm']):
        return 'Rain'
    if any(k in w for k in ['cloud', 'overcast', 'partly']):
        return 'Cloudy'
    if any(k in w for k in ['fair', 'clear', 'sunny']):
        return 'Clear'
    if any(k in w for k in ['wind', 'dust', 'sand']):
        return 'Wind/Dust'
    return 'Other'
df['Weather_Group'] = df['Weather_Condition'].apply(group_weather)


In [40]:
df['Has_Junction']  = df['Junction'].astype(bool)
df['Has_Signal']    = df['Traffic_Signal'].astype(bool)
df['Road_Feature']  = df.apply(
    lambda r: 'Junction' if r['Junction']
         else ('Signal'   if r['Traffic_Signal']
         else ('Crossing' if r['Crossing']
         else 'Open Road')), axis=1)

# COVID period flag (for filtering/annotation in Tableau)
df['COVID_Period'] = df['Year'].isin([2020, 2021])

7. FILTER OBVIOUS OUTLIERS

In [41]:
# Remove temperature extremes (sensor errors)
df = df[(df['Temperature(F)'] > -30) & (df['Temperature(F)'] < 130)]
# Remove implausible lat/lng
df = df[(df['Start_Lat'] > 24) & (df['Start_Lat'] < 50)]
df = df[(df['Start_Lng'] > -125) & (df['Start_Lng'] < -66)]

8. EXPORT CLEAN FILE

In [42]:
df.to_csv('us_accidents_clean.csv', index=False)
print(f"Clean shape: {df.shape}")
print(df[['Hour','DayOfWeek','Weather_Group','Severity_Group','Duration_Min']].head())

Clean shape: (6985057, 37)
   Hour DayOfWeek Weather_Group Severity_Group  Duration_Min
0     5    Monday          Rain     High (3–4)         314.0
1     6    Monday          Rain      Low (1–2)          30.0
2     6    Monday        Cloudy      Low (1–2)          30.0
3     7    Monday        Cloudy     High (3–4)          30.0
4     7    Monday        Cloudy      Low (1–2)          30.0


---

#### Sampled

In [43]:
# File 1: Full detail (for map and time-based charts — sample to 500K)
df_sample = df.sample(n=500_000, random_state=42)
df_sample.to_csv('accidents_detail.csv', index=False)

#### Aggregated

In [51]:
# File 2: Pre-aggregated (for bar/grouped charts — fast and complete)
agg = df.groupby(
    ['State', 'Year', 'Hour', 'Severity_Group',
     'Weather_Group', 'Time_Bucket', 'Is_Weekend', 'Road_Feature']
).agg(
    Accident_Count=('ID', 'count'),
    Avg_Duration=('Duration_Min', 'mean'),
    High_Sev_Count=('Severity_Group', lambda x: (x == 'High (3–4)').sum())
).reset_index()

agg['High_Sev_Rate'] = agg['High_Sev_Count'] / agg['Accident_Count']
agg.to_csv('accidents_aggregated.csv', index=False)
print(f"Aggregated shape: {agg.shape}")

Aggregated shape: (257062, 12)


In [52]:
import pandas as pd
agg = pd.read_csv('accidents_aggregated.csv')
print(agg.shape)

(257062, 12)


In [53]:
print(agg['Accident_Count'].sum())        # Should be close to 6.98M
print(agg['Severity_Group'].unique())     # Should show High (3–4) and Low (1–2)
print(agg['Weather_Group'].unique())      # Should show all 7 weather categories
print(agg['Time_Bucket'].unique())        # Should show 5 buckets

6985057
['High (3–4)' 'Low (1–2)']
['Clear' 'Cloudy' 'Fog/Haze' 'Other' 'Rain' 'Snow/Ice' 'Wind/Dust']
['Night (0–6)' 'AM Rush (6–10)' 'Midday (10–16)' 'PM Rush (16–20)'
 'Evening (20–24)']


In [54]:
import pandas as pd

files = {
    'us_accidents_clean.csv': 'us_accidents_clean.csv',
    'accidents_aggregated.csv': 'accidents_aggregated.csv',
    'accidents_detail.csv': 'accidents_detail.csv'
}

for name, path in files.items():
    df_temp = pd.read_csv(path)
    print(f"\n{'='*40}")
    print(f"📄 {name}")
    print(f"Shape: {df_temp.shape}")
    print(f"Columns: {list(df_temp.columns)}")
    print(f"Memory: {df_temp.memory_usage(deep=True).sum() / 1e6:.1f} MB")


📄 us_accidents_clean.csv
Shape: (6985057, 37)
Columns: ['ID', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'City', 'County', 'State', 'Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Weather_Condition', 'Wind_Speed(mph)', 'Junction', 'Traffic_Signal', 'Crossing', 'Roundabout', 'Sunrise_Sunset', 'Civil_Twilight', 'Distance(mi)', 'Hour', 'DayOfWeek', 'Month', 'Year', 'Is_Weekend', 'Month_Num', 'DayOfWeek_Num', 'Duration_Min', 'Time_Bucket', 'Is_Rush_Hour', 'Severity_Group', 'Weather_Group', 'Has_Junction', 'Has_Signal', 'Road_Feature', 'COVID_Period']
Memory: 7239.5 MB

📄 accidents_aggregated.csv
Shape: (257062, 12)
Columns: ['State', 'Year', 'Hour', 'Severity_Group', 'Weather_Group', 'Time_Bucket', 'Is_Weekend', 'Road_Feature', 'Accident_Count', 'Avg_Duration', 'High_Sev_Count', 'High_Sev_Rate']
Memory: 100.3 MB

📄 accidents_detail.csv
Shape: (500000, 37)
Columns: ['ID', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'City', 'County', 'State', 'Te

In [ ]:
# seperate file to get day of week
agg_heatmap = df.groupby(
    ['DayOfWeek', 'Hour', 'Severity_Group']
).agg(
    Accident_Count=('ID', 'count'),
    High_Sev_Count=('Severity_Group', lambda x: (x == 'High (3–4)').sum())
).reset_index()

agg_heatmap['High_Sev_Rate'] = agg_heatmap['High_Sev_Count'] / agg_heatmap['Accident_Count']
agg_heatmap.to_csv('accidents_heatmap.csv', index=False)
print(f"Heatmap shape: {agg_heatmap.shape}")

Heatmap shape: (336, 6)


In [56]:
# Total accidents per hour+day
total = df.groupby(['DayOfWeek', 'Hour']).size().reset_index(name='Total_Count')

# High severity accidents per hour+day
high = df[df['Severity_Group'] == 'High (3–4)'].groupby(['DayOfWeek', 'Hour']).size().reset_index(name='High_Count')

agg_heatmap = total.merge(high, on=['DayOfWeek', 'Hour'])
agg_heatmap['High_Sev_Rate'] = agg_heatmap['High_Count'] / agg_heatmap['Total_Count']

day_order = {'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7}
agg_heatmap['DayOfWeek_Num'] = agg_heatmap['DayOfWeek'].map(day_order)

agg_heatmap.to_csv('accidents_heatmap.csv', index=False)
print(agg_heatmap.shape)
print(agg_heatmap['High_Sev_Rate'].describe())

(168, 6)
count    168.000000
mean       0.213603
std        0.033199
min        0.139174
25%        0.192851
50%        0.211096
75%        0.231298
max        0.329883
Name: High_Sev_Rate, dtype: float64
